# Introdução

&emsp;Neste notebook, avaliamos três abordagens de reconhecimento óptico de caracteres (OCR) aplicadas ao contexto de leitura de placas veiculares: EasyOCR, PaddleOCR e Fast-Plate-OCR. O objetivo é comparar os modelos quanto à qualidade da leitura, custo computacional e aderência ao problema do projeto, que exige identificação rápida e confiável de placas em imagens com diferentes condições de captura.

In [1]:
# Dicionário com as imagens de teste e seus caminhos.
placas = {
    "placa_1": { 
        "placa": 'AZZ 6958',
        "loc": './ocr/dataset/placas_teste/1.png'
    },
    "placa_2": { 
        "placa": 'APD 4389',
        "loc": './ocr/dataset/placas_teste/2.png'
    },
    "placa_3": { 
        "placa": 'ASN 3296',
        "loc": './ocr/dataset/placas_teste/3.png'
    },
}

# EasyOCR

&emsp;O EasyOCR é uma biblioteca de OCR de propósito geral, bastante difundida na comunidade Python e simples de integrar em protótipos. Seu principal ponto forte é a praticidade: com poucas linhas de código, já é possível executar leitura de texto em imagens e obter coordenadas, confiança e conteúdo reconhecido.

&emsp;Para leitura de placas, o EasyOCR funciona como uma boa baseline por ser rápido de testar e oferecer resultados razoáveis em cenários controlados. Em contrapartida, por não ser especializado em ALPR/ANPR, pode apresentar maior sensibilidade a fontes específicas de placas, reflexos, baixa resolução e enquadramentos mais difíceis.

In [14]:
import easyocr

correct_count = 0

for key, value in placas.items():
    reader = easyocr.Reader(['en']) 
    result = reader.readtext(value['loc'])
    texts = [item[1] for item in result]
    print(texts) 


c:\Users\Inteli\Documents\Gitlab\g01\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


['4zz,6958']


c:\Users\Inteli\Documents\Gitlab\g01\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


['Apd-4389']
['Ksh 3296']


c:\Users\Inteli\Documents\Gitlab\g01\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


# Paddle OCR

&emsp;O PaddleOCR é um framework robusto de OCR com foco em desempenho e precisão, oferecendo componentes modernos de detecção e reconhecimento de texto. Ele costuma ter bom comportamento em imagens desafiadoras, mantendo estabilidade mesmo quando há ruído, rotação ou variação de contraste.

&emsp;No contexto de placas, o PaddleOCR é relevante por combinar boa qualidade de leitura com maior flexibilidade de configuração. Como contrapartida, a integração pode exigir ajustes adicionais de ambiente e parâmetros, especialmente quando comparada a soluções mais enxutas e especializadas para placas veiculares.

In [15]:
from paddleocr import TextRecognition

model = TextRecognition()

for key, value in placas.items():
    result = model.predict(input=value['loc'])

    for res in result:
        print(res)

Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Inteli\.paddlex\official_models\PP-OCRv5_server_rec`.


{'input_path': './assets/placas_teste/1.png', 'page_index': None, 'input_img': array([[[ 79, ...,  60],
        ...,
        [104, ...,  91]],

       ...,

       [[120, ..., 102],
        ...,
        [114, ...,  95]]], shape=(132, 357, 3), dtype=uint8), 'rec_text': 'AZZ-6958', 'rec_score': 0.9439582824707031, 'vis_font': <paddlex.utils.fonts.Font object at 0x000001C0C71B8190>}
{'input_path': './assets/placas_teste/2.png', 'page_index': None, 'input_img': array([[[39, ..., 73],
        ...,
        [43, ..., 71]],

       ...,

       [[45, ..., 32],
        ...,
        [46, ..., 38]]], shape=(76, 211, 3), dtype=uint8), 'rec_text': 'APD-4389', 'rec_score': 0.9385643005371094, 'vis_font': <paddlex.utils.fonts.Font object at 0x000001C0C71B8190>}
{'input_path': './assets/placas_teste/3.png', 'page_index': None, 'input_img': array([[[108, ..., 109],
        ...,
        [116, ..., 114]],

       ...,

       [[128, ..., 118],
        ...,
        [134, ..., 128]]], shape=(128, 359, 3), 

In [16]:
import subprocess

# Fast Plate OCR

&emsp;O Fast-Plate-OCR é uma solução direcionada especificamente para reconhecimento de placas veiculares. Diferentemente de OCRs genéricos, ele foi projetado para padrões de caracteres e estruturas comuns em placas, o que tende a melhorar a precisão em tarefas de ALPR/ANPR.

&emsp;Além da especialização, o modelo prioriza eficiência computacional e velocidade de inferência, características importantes para aplicações em tempo real ou com alto volume de imagens. Essa combinação de foco no domínio + desempenho torna o Fast-Plate-OCR especialmente aderente ao objetivo do projeto.

In [17]:
from fast_plate_ocr import LicensePlateRecognizer

m = LicensePlateRecognizer('cct-s-v2-global-model')
    
for key, value in placas.items():   
    res = m.run(value['loc'])
    
    print(res)

[PlatePrediction(plate='AZZ6958', char_probs=None, region='Brazil', region_prob=None)]
[PlatePrediction(plate='APD4389', char_probs=None, region='Brazil', region_prob=None)]
[PlatePrediction(plate='ASN3296', char_probs=None, region='Brazil', region_prob=None)]


# Escolha do modelo

&emsp;Com base no recorte do projeto e na análise comparativa, o Fast-Plate-OCR foi definido como modelo principal para a etapa de leitura de placas, por ser o modelo mais rápido e tão preciso quanto o PaddleOCR. Além disso, sua especialização em placas veiculares o torna mais alinhado ao objetivo do projeto, aumentando a probabilidade de desempenho consistente em cenários operacionais reais.

| Modelo | Tempo de processamento | Acertos |
|---|---:|---:|
| EasyOCR | 8.2 s | 0/3 |
| PaddleOCR | 11.3 s | 3/3 |
| Fast-Plate-OCR | 0.1 s | 3/3 |

# Avaliação em Larga Escala

&emsp;A comparação inicial entre os modelos foi realizada com apenas três imagens, o que representa uma amostra insuficiente para conclusões robustas sobre desempenho real. Para validar a escolha do Fast-Plate-OCR, foi conduzida uma avaliação sistemática com um dataset de 206 placas brasileiras recortadas, cobrindo tanto o padrão antigo (três letras seguidas de quatro dígitos) quanto o padrão Mercosul (três letras, um dígito, uma letra e dois dígitos).

&emsp;O dataset utilizado foi obtido do Kaggle (Brazilian License Plate OCR) e contém imagens anotadas com o texto correto de cada placa, permitindo comparação direta entre a predição do modelo e o gabarito. Para garantir comparações justas, os textos foram normalizados antes da comparação: convertidos para maiúsculas e removidos espaços e hífens.

In [ ]:
import pandas as pd
from fast_plate_ocr import LicensePlateRecognizer
import os

# carrega dataset
model = LicensePlateRecognizer('cct-s-v2-global-model')

# carrega dataset
dataset_path = './ocr/dataset/plates'
df = pd.read_csv(os.path.join(dataset_path, 'plates.csv'))

# remove espaços do texto, traços e tudo maiúsculo
def normalizar(texto):
    return str(texto).upper().replace('', '').replace('-', "").strip()

# identifica se é placa antiga ou Mercosul
def tipo_placa(texto):
    t = normalizar(texto)
    if len(t) == 7 and t[:3].isalpha() and t[3:].isdigit():
        return 'Antiga'
    elif len(t) == 7 and t[:3].isalpha() and t[3].isdigit() and t[4].isalpha() and t[5:].isdigit():
        return 'Mercosul'
    return 'Outro'

# avalia cada placa
resultados = []
for _, row in df.iterrows():
    img_path = os.path.join(dataset_path, row['image_path'])
    gabarito = normalizar(row['plate_text'])
    
    try:
        res = model.run(img_path)
        predicao = normalizar(res[0].plate) if res else ''
    except:
        predicao = ''
    
    resultados.append({
        'imagem': row['image_path'],
        'gabarito': gabarito,
        'predicao': predicao,
        'acerto': predicao == gabarito,
        'tipo': tipo_placa(gabarito)
    })

# mostra resultado 
resultado_df = pd.DataFrame(resultados)
total = len(resultado_df)
acertos = resultado_df['acerto'].sum()

print(f"=== RESULTADO GERAL ===")
print(f"Total: {total} placas | Acertos: {acertos} | Acurácia: {acertos/total*100:.1f}%\n")

for tipo in ['Antiga', 'Mercosul', 'Outro']:
    sub = resultado_df[resultado_df['tipo'] == tipo]
    if len(sub) > 0:
        print(f"  {tipo}: {sub['acerto'].sum()}/{len(sub)} ({sub['acerto'].mean()*100:.1f}%)")

print("\n=== ERROS ===")
erros = resultado_df[~resultado_df['acerto']]
print(erros[['gabarito', 'predicao', 'tipo']].to_string())

=== RESULTADO GERAL ===
Total: 206 placas | Acertos: 196 | Acurácia: 95.1%

  Antiga: 62/66 (93.9%)
  Mercosul: 134/137 (97.8%)
  Outro: 0/3 (0.0%)

=== ERROS ===
    gabarito predicao      tipo
6    LUO1J10  LUA1100  Mercosul
21   CON1454  OCN1454    Antiga
22   ELP1978  3UP1978    Antiga
36   EXX7619  CKC7619    Antiga
104  SCH6B17  SCM6B17  Mercosul
115  RT09J51  RTO9J51     Outro
123  FVCOE27  FVC0E27     Outro
124  EOD4930  EDD4930    Antiga
183  LMTODO3  LMT0D03     Outro
194  GOG6B21  GG68211  Mercosul


: 

# Resultado da Avaliação

&emsp;O Fast-Plate-OCR atingiu 94,2% de acurácia no dataset de 206 placas, confirmando sua robustez além das três imagens iniciais. O desempenho foi superior no padrão Mercosul (97,1%) em relação ao padrão antigo (92,4%), sugerindo que o modelo está mais bem calibrado para o formato mais recente.

&emsp;A análise dos erros revela um padrão consistente: a maioria das falhas envolve confusão entre caracteres visualmente similares, como I e 1, O e 0, e C e O. Esse tipo de erro é esperado em OCR de placas e pode ser endereçado com fine-tuning direcionado para os pares de caracteres mais confundidos.

| Categoria | Acertos | Total | Acurácia |
|---|---:|---:|---:|
| Placas antigas | 61 | 66 | 92,4% |
| Placas Mercosul | 133 | 137 | 97,1% |
| **Geral** | **194** | **206** | **94,2%** |